# Эмбеддинги при помощи обычной cnn

In [10]:
import pandas as pd

In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2
from PIL import Image

In [13]:
from tqdm import tqdm

In [2]:
def preprocess(img_path):
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    img = Image.open(img_path).convert('RGB')
    return transform(img).unsqueeze(0)

In [3]:
model = mobilenet_v2(pretrained=True)
model.classifier = torch.nn.Identity()
model.eval()

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /home/dev_ds_common/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13.6M/13.6M [00:00<00:00, 27.9MB/s]


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [4]:
def get_embedding(img_path):
    with torch.no_grad():
        processed_image = preprocess(img_path)
        embedding = model(processed_image)
    return embedding

In [6]:
img_path = '/cv-sandbox/project/data/IMG_20220413_110954.jpg'
embedding = get_embedding(img_path).detach().cpu().numpy().reshape(-1)

In [7]:
embedding.shape

(1, 1280)

**К уже построенным эмбеддингам DINOv2 добавляем эмбеддинги Mobilenet v2**

In [11]:
markup_df = pd.read_parquet('/cv-sandbox/project/image_embeddings.pq')

In [12]:
markup_df.head(3)

,image_path,embedding,embedding_clf
0,/cv-sandbox/project/data/IMG_20221201_131637.jpg,"[-2.8007195, 0.1795005, 2.0634825, 0.31013796,...","[-1.198553, -0.61378217, -0.49053022, -1.18711..."
1,/cv-sandbox/project/data/IMG_20220422_185636.jpg,"[-0.39972767, 0.23981501, 1.5460123, -1.038316...","[-3.2148683, -1.8561819, -3.4543087, -2.958324..."
2,/cv-sandbox/project/data/P7081020.JPG,"[0.8322267, 0.17672215, -0.3619432, 0.58000106...","[-1.4701422, 0.23119043, -1.671558, 1.1484444,..."


In [15]:
embeddings = []
for pth in tqdm(markup_df['image_path'].values):
    embeddings.append(get_embedding(pth).detach().cpu().numpy())

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 299/299 [00:09<00:00, 32.17it/s]


In [16]:
markup_df['embedding_cnn'] = embeddings

In [17]:
markup_df.head(3)

,image_path,embedding,embedding_clf,embedding_cnn
0,/cv-sandbox/project/data/IMG_20221201_131637.jpg,"[-2.8007195, 0.1795005, 2.0634825, 0.31013796,...","[-1.198553, -0.61378217, -0.49053022, -1.18711...","[[0.30277967, 0.7905779, 0.294772, 0.8011666, ..."
1,/cv-sandbox/project/data/IMG_20220422_185636.jpg,"[-0.39972767, 0.23981501, 1.5460123, -1.038316...","[-3.2148683, -1.8561819, -3.4543087, -2.958324...","[[0.024153264, 0.14707933, 0.0070938347, 0.089..."
2,/cv-sandbox/project/data/P7081020.JPG,"[0.8322267, 0.17672215, -0.3619432, 0.58000106...","[-1.4701422, 0.23119043, -1.671558, 1.1484444,...","[[0.23040602, 0.38108802, 0.16922201, 0.514281..."


In [ ]:
markup_df.to_parquet('/cv-sandbox/project/image_embeddings.pq')